# Analyse av Markedsdata

## Imports

In [99]:
from dotenv import load_dotenv
import os

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

In [100]:
# henter keys
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

## Loading data

In [101]:
client = StockHistoricalDataClient(KEY, SECRET)

bars = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols = "AAPL",
    timeframe = TimeFrame.Day,
    start = "2026-01-01",
))

df = bars.df
df.info()

<class 'pandas.DataFrame'>
MultiIndex: 115 entries, ('AAPL', Timestamp('2026-01-02 05:00:00+0000', tz='UTC')) to ('AAPL', Timestamp('2026-06-17 04:00:00+0000', tz='UTC'))
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         115 non-null    float64
 1   high         115 non-null    float64
 2   low          115 non-null    float64
 3   close        115 non-null    float64
 4   volume       115 non-null    float64
 5   trade_count  115 non-null    float64
 6   vwap         115 non-null    float64
dtypes: float64(7)
memory usage: 7.5+ KB


In [102]:
df.describe()

,open,high,low,close,volume,trade_count,vwap
count,115.000000,115.000000,115.000000,115.000000,1.150000e+02,1.150000e+02,115.000000
mean,272.104635,275.102763,269.238537,272.148870,4.779247e+07,7.228712e+05,272.168641
std,19.039769,19.420101,19.017519,19.121707,1.443943e+07,1.627428e+05,19.218418
min,247.320000,249.199900,243.420000,246.630000,2.637259e+07,4.825740e+05,246.869425
25%,258.025000,260.195000,255.495000,258.555000,3.882886e+07,6.156180e+05,257.949283
50%,266.800000,269.430000,262.450000,266.170000,4.462340e+07,6.986630e+05,265.602793
75%,285.592500,290.080000,283.425000,287.475000,5.214169e+07,7.785220e+05,287.572015
max,314.175000,317.400000,309.650000,315.200000,9.244341e+07,1.333467e+06,313.143117


## Cleaning

In [103]:
print(df.isnull().sum())
print(df.isnull().sum().sum())

open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64
0


Rydding er derfor ikke nødvendig, men en lett måte å fjerne manglende data er å slette hele raden:
```python
df.dropna()
```

In [104]:
# Ser etter om data er tidsserie
df.index

MultiIndex([('AAPL', '2026-01-02 05:00:00+00:00'),
            ('AAPL', '2026-01-05 05:00:00+00:00'),
            ('AAPL', '2026-01-06 05:00:00+00:00'),
            ('AAPL', '2026-01-07 05:00:00+00:00'),
            ('AAPL', '2026-01-08 05:00:00+00:00'),
            ('AAPL', '2026-01-09 05:00:00+00:00'),
            ('AAPL', '2026-01-12 05:00:00+00:00'),
            ('AAPL', '2026-01-13 05:00:00+00:00'),
            ('AAPL', '2026-01-14 05:00:00+00:00'),
            ('AAPL', '2026-01-15 05:00:00+00:00'),
            ...
            ('AAPL', '2026-06-04 04:00:00+00:00'),
            ('AAPL', '2026-06-05 04:00:00+00:00'),
            ('AAPL', '2026-06-08 04:00:00+00:00'),
            ('AAPL', '2026-06-09 04:00:00+00:00'),
            ('AAPL', '2026-06-10 04:00:00+00:00'),
            ('AAPL', '2026-06-11 04:00:00+00:00'),
            ('AAPL', '2026-06-12 04:00:00+00:00'),
            ('AAPL', '2026-06-15 04:00:00+00:00'),
            ('AAPL', '2026-06-16 04:00:00+00:00'),
            ('A

In [105]:
df.index.get_level_values("timestamp")

DatetimeIndex(['2026-01-02 05:00:00+00:00', '2026-01-05 05:00:00+00:00',
               '2026-01-06 05:00:00+00:00', '2026-01-07 05:00:00+00:00',
               '2026-01-08 05:00:00+00:00', '2026-01-09 05:00:00+00:00',
               '2026-01-12 05:00:00+00:00', '2026-01-13 05:00:00+00:00',
               '2026-01-14 05:00:00+00:00', '2026-01-15 05:00:00+00:00',
               ...
               '2026-06-04 04:00:00+00:00', '2026-06-05 04:00:00+00:00',
               '2026-06-08 04:00:00+00:00', '2026-06-09 04:00:00+00:00',
               '2026-06-10 04:00:00+00:00', '2026-06-11 04:00:00+00:00',
               '2026-06-12 04:00:00+00:00', '2026-06-15 04:00:00+00:00',
               '2026-06-16 04:00:00+00:00', '2026-06-17 04:00:00+00:00'],
              dtype='datetime64[us, UTC]', name='timestamp', length=115, freq=None)

Alpaca git allerede en tidsserie data, men i et tilfelle der de ikke gjør det kan du gjøre dette:
```python
# 1. Konverter datokolonnen til ekte datetime-objekter
df['dato'] = pd.to_datetime(df['dato'])

# 2. Sett datokolonnen som indeks i tabellen
df = df.set_index('dato')

# 3. Sorter indeksen kronologisk (viktig for tidsserier!)
df = df.sort_index()
```

## Analyse

In [106]:
import plotly.express as px
import plotly.graph_objects as go

In [107]:
# Korrelasjonsmatrise
df.corr() # men dette er ikke så veldig visuelt

,open,high,low,close,volume,trade_count,vwap
open,1.000000,0.992425,0.990281,0.980711,0.112989,0.393187,0.987957
high,0.992425,1.000000,0.992021,0.991031,0.144715,0.419570,0.996179
low,0.990281,0.992021,1.000000,0.994017,0.074811,0.335162,0.997494
close,0.980711,0.991031,0.994017,1.000000,0.091323,0.348835,0.997796
volume,0.112989,0.144715,0.074811,0.091323,1.000000,0.773859,0.102332
trade_count,0.393187,0.419570,0.335162,0.348835,0.773859,1.000000,0.366360
vwap,0.987957,0.996179,0.997494,0.997796,0.102332,0.366360,1.000000


In [108]:
# Heatmap korrelasjonsmatrise for å se hvilke variabler beskriver hverandre

corr = df.corr()

px.imshow(
    corr,
    zmin= -1,
    zmax= 1,
    text_auto= ".2f",
    color_continuous_scale = "RdBu_r"

).show()

**Skulle jeg ha valgt noen dominerende features ville jeg ha valgt å jobbe med low og volume**

In [109]:
# Linjeplot for å observere trender, at den går oppover eller nedover
dfi = df.reset_index()

px.line(
    dfi,
    x = "timestamp",
    y = "close"
).show()

**Vi ser at grafen hadde tydelig opptrend fra april til juni, deretter mindre korreksjon.**

In [110]:
# histogram for se om store tap er vanlig, observere ekstreme hendelser og om avkastningen er normalfordelt
returns = dfi["close"].pct_change().dropna()

px.histogram(
    returns,
    nbins = 5
).show()

**De fleste daglige avkastningene er små, med noen større positive og negative bevegelser**

| Del                     | Hva er det?                          | Hva forteller det?                          | Tolkning                                                           |
| ----------------------- | ------------------------------------ | ------------------------------------------- | ------------------------------------------------------------------ |
| **X-akse**              | Percent change (%)                   | Viser størrelsen på kursendringene          | Negative verdier = tap, positive verdier = gevinst                 |
| **Y-akse**              | Frekvens (antall observasjoner)      | Hvor ofte ulike prosentendringer forekommer | Høye søyler betyr vanlige endringer                                |
| **Søyler (bins)**       | Grupper av prosentendringer          | Viser fordelingen av avkastning             | Brede eller smale avhengig av antall intervaller                   |
| **Topp (peak)**         | Den høyeste søylen                   | Den mest vanlige prosentendringen           | Kalles ofte "typisk" avkastning                                    |
| **Spredning**           | Hvor bredt histogrammet strekker seg | Hvor mye avkastningen varierer              | Stor spredning = høy volatilitet                                   |
| **Haler (tails)**       | Endene av histogrammet               | Sjeldne, ekstreme bevegelser                | Lange haler = høy risiko for store utslag                          |
| **Skjevhet (skewness)** | Asymmetri i fordelingen              | Om store gevinster eller tap er mer vanlige | Høyreskjev = flere store gevinster; venstreskjev = flere store tap |

<br>

| Situasjon                    | Hva betyr det?                       |
| ---------------------------- | ------------------------------------ |
| Histogram sentrert rundt 0 % | Kursen beveger seg vanligvis lite    |
| Smalt histogram              | Lav volatilitet                      |
| Bredt histogram              | Høy volatilitet                      |
| Lang hale mot venstre        | Risiko for store tap                 |
| Lang hale mot høyre          | Mulighet for store gevinster         |
| To topper (bimodalt)         | To ulike markedsregimer eller atferd |

In [111]:
# Box for å se hvor stor den daglige avkastningen er, har aksjen mange ekstreme dager

px.box(
    returns
).show()

**Flere eksteme tap dager enn gevinst dager**

| Del                       | Hva er det?                | Hva forteller det?                         | Tolkning                                                     |
| ------------------------- | -------------------------- | ------------------------------------------ | ------------------------------------------------------------ |
| **Median**                | Streken inne i boksen      | Den midterste observasjonen                | Typisk daglig avkastning                                     |
| **Boksen (IQR)**          | Fra 25- til 75-persentil | De midterste 50 % av observasjonene        | Viser normal variasjon                                       |
| **Nedre kvartil (Q1)**    | Nederste del av boksen     | 25 % av observasjonene ligger under        | Grense for lavere "normale" verdier                          |
| **Øvre kvartil (Q3)**     | Øverste del av boksen      | 75 % av observasjonene ligger under        | Grense for høyere "normale" verdier                          |
| **Whiskers**              | Linjer utenfor boksen      | Typisk minimum og maksimum uten uteliggere | Viser vanlig variasjon                                       |
| **Outliers** | Punkter utenfor whiskers   | Uvanlige prosentendringer                  | Kan skyldes nyheter, kriser eller ekstreme markedsbevegelser |

<br>

| Situasjon                    | Hva betyr det?                         |
| ---------------------------- | -------------------------------------- |
| Liten boks                   | Lav volatilitet                        |
| Stor boks                    | Høy volatilitet                        |
| Median nær midten av boksen  | Symmetrisk fordeling                   |
| Median nær toppen av boksen  | Flere eller større negative bevegelser |
| Median nær bunnen av boksen  | Flere eller større positive bevegelser |
| Mange uteliggere             | Hyppige ekstreme kursbevegelser        |
| Flere uteliggere på nedsiden | Høy risiko for store tap               |
| Flere uteliggere på oppsiden | Potensial for store gevinster          |

In [112]:
# canclestick for å se hvordan prisutviklingen endrer seg
go.Figure(go.Candlestick(
    x = dfi["timestamp"],
    open = dfi["open"],
    close = dfi["close"],
    high = dfi["high"],
    low = dfi["low"]
)).show()

**Flere større røde candles som tyder på at selgere har mer kontroll over prisutviklingen**

**Hva er den candlestick**
| Del / Type                    | Hva er det?                                             | Hva forteller det?                                | Vanlig tolkning                                                             |
| ----------------------------- | ------------------------------------------------------- | ------------------------------------------------- | --------------------------------------------------------------------------- |
| **Kropp (Body)**              | Den tykke delen av candlen mellom åpnings- og sluttkurs | Viser hvem som vant perioden                      | Stor kropp = sterk bevegelse. Liten kropp = usikkerhet                      |
| **Åpningskurs (Open)**        | Kursen ved starten av perioden                          | Utgangspunktet for handelen                       | Sammenlignes med sluttkursen                                                |
| **Sluttkurs (Close)**         | Kursen ved slutten av perioden                          | Viser hvem som hadde kontroll ved periodens slutt | Regnes ofte som den viktigste kursen                                        |
| **Øvre shadow (Upper Wick)**  | Streken over kroppen                                    | Viser periodens høyeste kurs                      | Lang øvre shadow = kjøperne presset opp, men selgerne tok tilbake kontroll  |
| **Nedre shadow (Lower Wick)** | Streken under kroppen                                   | Viser periodens laveste kurs                      | Lang nedre shadow = selgerne presset ned, men kjøperne tok tilbake kontroll |


**Ulike Candle-typer**
| Candle-type | Kjennetegn              | Hva forteller den?               |
| ----------- | ----------------------- | -------------------------------- |
| Bullish     | Sluttkurs > åpningskurs | Kjøperne dominerer               |
| Bearish     | Sluttkurs < åpningskurs | Selgerne dominerer               |
| Doji        | Åpningskurs ≈ sluttkurs | Usikkerhet og balanse i markedet |

**Tolkning av lange whicks**
| Type shadow           | Hva skjedde?                                                           | Hva betyr det?           | Vanlig signal                                    |
| --------------------- | ---------------------------------------------------------------------- | ------------------------ | ------------------------------------------------ |
| **Lang øvre shadow**  | Kjøperne presset kursen høyt opp, men selgerne presset den ned igjen   | Høyere priser ble avvist | Potensielt **bearish**, særlig etter en opptrend |
| **Lang nedre shadow** | Selgerne presset kursen kraftig ned, men kjøperne løftet den opp igjen | Lavere priser ble avvist | Potensielt **bullish**, særlig etter en nedtrend |

